In [110]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
import zipfile

In [101]:
%run ./ADLSClient.ipynb

In [122]:
adls_client = ADLSClient()
adls_container_client = adls_client.create_container_client()

In [146]:
def download_and_upload(url, adls_container_client, adls_directory, latest_downloaded_file):
    try:
        print(f"Retrieving zip file from {url}.")
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        zip_file = BytesIO(response.content)
        print(f"{url} has been successfully stored in memory.")

        with zipfile.ZipFile(zip_file) as zf:
            for file_name in zf.namelist():
                if file_name.endswith('/'):
                    continue

                with zf.open(file_name) as file_data:
                    upload_path = f"{adls_directory}/{url.split('/')[-2]}/{file_name}"
                    print(f"Uploading {file_name} to ADLS at {upload_path}.")
                    
                    blob_client = adls_container_client.get_blob_client(upload_path)
                    blob_client.upload_blob(file_data.read(), overwrite=True)
                    print(f"Uploaded {file_name} successfully.")
    
    except requests.exceptions.RequestException as e:
        print(f"Failed to download ZIP file: {e}")
        
    except zipfile.BadZipFile:
        print("The downloaded file is not a valid ZIP file.")
        
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
url = "https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/"
html = requests.get(url).content
soup = BeautifulSoup(html, "html.parser")

links = [a["href"] for a in soup.find_all("a", href=True)]
links = [f"{url}{link}" for link in links if ".zip" in link]

blob_list = ['/'.join(blob.name.split('/')[-2:]) for blob in container_client.list_blobs() 
             if "fhwa_import/" in blob.name 
             and "Delimited_CleanData" not in blob.name]

latest_ingested_file = max(blob_list)
urls = [f"{url}{link}" for link in links if re.findall(r"/(\d{4})/", link)[0] >= re.findall(r"(\d{4})", latest_ingested_file)[0]]

for url in urls:
    download_and_upload(url, adls_container_client, adls_dir, latest_downloaded_file)

Retrieving zip file from https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2010/2010_station_data.zip.
https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2010/2010_station_data.zip has been successfully stored in memory.
Uploading Station_Data_Extract_Pipe_Delimited_CleanData_2010.txt to ADLS at fhwa_import/2010/Station_Data_Extract_Pipe_Delimited_CleanData_2010.txt.
Uploaded Station_Data_Extract_Pipe_Delimited_CleanData_2010.txt successfully.
Retrieving zip file from https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2011/2011_station_data.zip.
https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2011/2011_station_data.zip has been successfully stored in memory.
Uploading Station_Data_Extract_Pipe_Delimited_CleanData_2011.txt to ADLS at fhwa_import/2011/Station_Data_Extract_Pipe_Delimited_CleanData_2011.txt.
Uploaded Station_Data_Extract_Pipe_Delimited_CleanData_2011.txt successfully.
Retrieving zip file from https://www.fhwa.dot.gov/policyinformation/ta